# RSNA Knee MRI — EDA and Working Starter

Everything you need to get moving in this competition: what the data looks like, how to actually load it, a working DICOM pipeline, and a valid baseline `submission.csv` — plus three setup issues that cost me several failed runs, documented here so they don't cost you any.

## ⚠️ Issue 1 — the data is not at the usual path

This competition mounts under a **new layout**: `/kaggle/input/competitions/rsna-knee-abnormality-detection/` (note the extra `competitions/` level). Hardcoding the classic `/kaggle/input/<slug>/` gives `FileNotFoundError`. The robust fix is a glob (used below), which works in both layouts.

## ⚠️ Issue 2 — the P100 GPU no longer works with the default PyTorch

The current PyTorch build in Kaggle's image has dropped support for the Tesla **P100** (compute capability 6.0) — the first `.cuda()` op throws `CUDA error: no kernel image is available for execution on the device`. **Select the T4 x2 accelerator, not P100**, in Notebook Settings (or pass `--accelerator NvidiaTeslaT4` if pushing via the CLI/API).

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# Issue 1 fix: locate the data wherever it's mounted
PATH = os.path.dirname(glob.glob('/kaggle/input/**/train.csv', recursive=True)[0])
print('data root:', PATH)

train = pd.read_csv(os.path.join(PATH, 'train.csv'))
series = pd.read_csv(os.path.join(PATH, 'train_series.csv'))
LABELS = list(train.columns[2:14])
print(f'{len(train)} studies, {len(series)} series, {len(LABELS)} target labels')

## The task

One row per **study** (a knee MRI exam). Each study has several **series** (different pulse sequences / anatomical planes), stored as DICOM slices. We predict 12 independent probabilities per study — one per abnormality — scored by **macro-averaged AUC**.

A detail worth noticing: `train.csv` contains a `Report` column with the radiology report (in one of ~9 languages) for every **training** study. The hidden test set won't have reports — as Issue 3 below explains, these reports are not a bonus, they're where your training labels have to come from.

In [ ]:
prev = train[LABELS].mean().sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(prev.index, prev.values * 100, color='#4477AA')
for i, v in enumerate(prev.values):
    ax.text(v*100 + 0.5, i, f'{v*100:.0f}%', va='center', fontsize=9)
ax.set_xlabel('prevalence (%)'); ax.set_title(f'Label prevalence (n={int(train[LABELS].notna().all(axis=1).sum())} annotated studies)')
plt.tight_layout(); plt.show()

Labels range from ~15% (MCL) to ~60% (Effusion) — no extreme rarity, which is friendly for AUC.

## ⚠️ Issue 3 — only ~58 of 4,407 studies actually have labels

This is the big one. Run `train[LABELS].isna().sum()` and you'll find **~99% of the label cells are NaN** — only a small annotated subset carries the 12 targets. Pandas `mean()` silently skips NaN, so naive prevalence plots (including an earlier version of this one) look like they cover the full corpus when they don't. Practical consequences:

- You **cannot** train directly on `train.csv` labels at scale — a naive `BCEWithLogitsLoss` on raw label arrays will propagate NaN and silently destroy training.
- The intended path is **deriving targets from the `Report` column** (present for all training studies, absent at test time) — via multilingual rule extraction or an LLM pass — then training images against those derived labels, holding out the annotated subset as a clean validation set.
- Reports are multilingual (my rough count: ~37% English, then Spanish, Greek, Cyrillic-script, German, Croatian/Serbian, Turkish, Dutch, French) and ~130 studies share byte-identical template reports — keep identical-report groups on the same side of your train/val split or you leak.

In [ ]:
co = train[LABELS].corr()
fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(co, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS, fontsize=8)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, f'{co.iloc[i,j]:.2f}', ha='center', va='center', fontsize=6)
plt.colorbar(im, shrink=0.8); ax.set_title('Label correlations')
plt.tight_layout(); plt.show()

The osteoarthritis cluster (Medial/Lateral/PF OA) and Effusion↔Synovitis correlate strongly — a multi-label head sharing a backbone should exploit this for free.

In [ ]:
print('series per study:')
print(series.groupby('StudyInstanceUID').size().describe().round(1))
comp = series.groupby(['Anatomical_Plane', 'Fluid_Sensitive']).size().unstack()
comp.plot(kind='bar', figsize=(7, 3.5), color=['#BBBBBB', '#4477AA'])
plt.title('Series composition: plane x fluid-sensitivity'); plt.ylabel('series count')
plt.legend(['not fluid-sensitive', 'fluid-sensitive']); plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

has_sagfs = series[(series['Anatomical_Plane']=='Sagittal') & (series['Fluid_Sensitive']==1)]['StudyInstanceUID'].nunique()
print(f'studies with a sagittal fluid-sensitive series: {has_sagfs}/{train.StudyInstanceUID.nunique()}')

## Loading DICOM volumes — working code

Median 5 series per study. Sagittal fluid-sensitive series (present in nearly every study) are the radiologist's workhorse for ACL/meniscus/effusion — a strong single-series choice for a first model. Below: a loader that reads a series, orders slices by `InstanceNumber`, windows intensities by percentile, and samples a fixed number of slices.

In [ ]:
import pydicom
import cv2

def load_volume(study_uid, series_uid, n_slices=16, size=224, split='train_series'):
    sdir = os.path.join(PATH, split, study_uid, series_uid)
    slices = []
    for f in glob.glob(os.path.join(sdir, '*.dcm')):
        ds = pydicom.dcmread(f)
        slices.append((int(getattr(ds, 'InstanceNumber', 0)), ds.pixel_array.astype(np.float32)))
    slices.sort(key=lambda x: x[0])
    take = np.linspace(0, len(slices)-1, n_slices).round().astype(int)
    out = np.zeros((n_slices, size, size), dtype=np.uint8)
    for i, t in enumerate(take):
        img = slices[t][1]
        lo, hi = np.percentile(img, [1, 99])
        img = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
        out[i] = (cv2.resize(img, (size, size)) * 255).astype(np.uint8)
    return out

# visualize one study's sagittal fluid-sensitive series
ex_study = has_ex = series[(series['Anatomical_Plane']=='Sagittal') & (series['Fluid_Sensitive']==1)].iloc[0]
vol = load_volume(ex_study['StudyInstanceUID'], ex_study['SeriesInstanceUID'])
fig, axes = plt.subplots(2, 8, figsize=(16, 4.2))
for i, ax in enumerate(axes.flat):
    ax.imshow(vol[i], cmap='gray'); ax.axis('off')
row = train[train['StudyInstanceUID']==ex_study['StudyInstanceUID']]
pos = [c for c in LABELS if row.iloc[0][c]==1] if len(row) else []
plt.suptitle(f"Sagittal fluid-sensitive, 16 sampled slices | positives: {', '.join(pos) or 'none'}")
plt.tight_layout(); plt.show()

In [ ]:
# slice counts and native resolutions across a sample of series
rng = np.random.RandomState(0)
sample = series.sample(30, random_state=0)
stats = []
for _, r in sample.iterrows():
    files = glob.glob(os.path.join(PATH, 'train_series', r['StudyInstanceUID'], r['SeriesInstanceUID'], '*.dcm'))
    if files:
        ds = pydicom.dcmread(files[0])
        stats.append({'n_slices': len(files), 'rows': ds.Rows, 'cols': ds.Columns,
                      'plane': r['Anatomical_Plane']})
sdf = pd.DataFrame(stats)
print(sdf.groupby('plane')[['n_slices','rows','cols']].median())
print('\nslice count range:', sdf.n_slices.min(), '-', sdf.n_slices.max())

## Valid baseline submission

Constant per-label train prevalence — scores 0.5 AUC by construction, but validates the end-to-end hidden-test pipeline (always do this before investing in models).

In [ ]:
test = pd.read_csv(os.path.join(PATH, 'test.csv'))
sub = pd.DataFrame({'StudyInstanceUID': test['StudyInstanceUID']})
for c in LABELS:
    sub[c] = train[c].mean()
sub.to_csv('submission.csv', index=False)
sub

## Modeling roadmap (what I'm building on top of this)

0. **Label derivation first** (see Issue 3): extract the 12 targets from the multilingual reports; validate the extractor against the annotated subset.
1. **2.5D CNN**: sagittal fluid-sensitive series → 16 slices → pretrained 2D backbone per slice → pool → 12 sigmoid heads trained on derived labels. Simple, fast, strong baseline.
2. **Multi-series fusion**: axial + coronal towers for the OA / patellofemoral labels that sagittal alone under-serves.
3. **Report distillation**: use the train-only Spanish reports to train a text teacher, distill into the image model.
4. Per-label attention pooling over slices (different abnormalities live on different slices).

If this saved you time, an upvote helps others find it. Questions welcome in the comments — happy to help debug any of the three issues.